# Equal-weight portfolio: main Granger regression

This notebook constructs one fixed 20-stock equal-weight portfolio and tests whether hourly `AI_Risk_total` predicts its subsequent hourly return.

- The target portfolio is rebalanced to 1/20 per stock at the start of every regular 10:00–16:00 New York window.
- Portfolio simple return is the arithmetic mean of the 20 constituent `ret_oc` values; portfolio log return is `log1p(equal_weight_simple_return)`.
- A portfolio hour is valid only when all 20 constituent returns are valid. Missing premarket returns are never treated as zero and the portfolio composition is never changed silently.
- Return lag order $q$ is selected once from 1–3 by BIC on a common return-only sample.
- Risk lag orders $p=1,2,3$ use a common $p=3$-eligible sample.
- Controls are New York target-hour and weekday fixed effects. Trading-date clustered inference is primary; Newey–West HAC (`maxlags=5` observations) is reported as a sensitivity check.

Risk from $[t-1,t)$ or earlier predicts portfolio return during $[t,t+1)$. Contemporaneous risk is never used. Granger causality here means predictive content, not structural causality.

In [1]:
from pathlib import Path
import platform
import sys
import warnings

import numpy as np
import pandas as pd
import scipy
import statsmodels
import statsmodels.api as sm
from IPython.display import Markdown, display
from scipy import stats
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("default")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.6g}")

ROOT = Path.cwd().resolve()
if not (ROOT / "download_hourly_returns.py").exists():
    raise RuntimeError("Run this notebook from the project root directory.")

REGULAR_PATH = ROOT / "data" / "hourly_10_16" / "hourly_returns_long.csv"
BRIDGE_PATH = ROOT / "data" / "premarket_bridge_07_10" / "hourly_returns_long.csv"
RISK_PATH = ROOT / "data" / "hourly_ai_risk_index" / "x" / "hourly_ai_risk_index.csv"

for path in (REGULAR_PATH, BRIDGE_PATH, RISK_PATH):
    if not path.exists():
        raise FileNotFoundError(path)

environment = pd.DataFrame(
    {
        "component": ["Python", "pandas", "NumPy", "SciPy", "statsmodels", "platform"],
        "version": [
            platform.python_version(),
            pd.__version__,
            np.__version__,
            scipy.__version__,
            statsmodels.__version__,
            platform.platform(),
        ],
    }
)
display(environment)
print("Interpreter:", sys.executable)
print("Project root:", ROOT)

,component,version
0,Python,3.14.7
1,pandas,3.0.6
2,NumPy,2.5.3
3,SciPy,1.18.1
4,statsmodels,0.15.0
5,platform,Windows-11-10.0.26200-SP0


Interpreter: C:\Users\Jerry Cai\Desktop\NLP_Assignment4_AI_risk_and_AI_development\.venv\Scripts\python.exe
Project root: C:\Users\Jerry Cai\Desktop\NLP_Assignment4_AI_risk_and_AI_development


## 1. Load, validate and align the source data

In [2]:
def parse_bool(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip().str.lower().eq("true")


def load_return_file(path: Path, expected_mode: str) -> pd.DataFrame:
    frame = pd.read_csv(path)
    required = {
        "ticker", "trading_date", "bar_start_utc", "bar_end_utc",
        "duration_minutes", "window_complete", "ohlc_valid", "nonpositive_price",
        "split_event", "log_ret_oc", "return_requires_review",
    }
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f"{path} is missing columns: {sorted(missing)}")
    frame["bar_start_utc"] = pd.to_datetime(frame["bar_start_utc"], utc=True)
    frame["bar_end_utc"] = pd.to_datetime(frame["bar_end_utc"], utc=True)
    frame["window_complete"] = parse_bool(frame["window_complete"])
    frame["ohlc_valid"] = parse_bool(frame["ohlc_valid"])
    frame["nonpositive_price"] = parse_bool(frame["nonpositive_price"])
    frame["split_event"] = parse_bool(frame["split_event"])
    frame["return_requires_review"] = parse_bool(frame["return_requires_review"])
    frame["duration_minutes"] = pd.to_numeric(frame["duration_minutes"], errors="coerce")
    frame["log_ret_oc"] = pd.to_numeric(frame["log_ret_oc"], errors="coerce")
    if "window_mode" in frame.columns and set(frame["window_mode"].dropna().unique()) != {expected_mode}:
        raise ValueError(f"Unexpected window mode in {path}")
    if frame.duplicated(["ticker", "bar_start_utc"]).any():
        raise ValueError(f"Duplicate ticker/start timestamps in {path}")
    return frame


regular = load_return_file(REGULAR_PATH, "clock_10_16")
bridge = load_return_file(BRIDGE_PATH, "bridge_07_10")
risk = pd.read_csv(RISK_PATH)

risk_required = {
    "hour", "AI_Risk_total", "coverage_status", "coverage_complete",
    "is_backtest_eligible_observed_sample", "is_backtest_eligible_strict",
}
missing = risk_required.difference(risk.columns)
if missing:
    raise ValueError(f"Risk index is missing columns: {sorted(missing)}")

risk["hour"] = pd.to_datetime(risk["hour"], utc=True)
risk["AI_Risk_total"] = pd.to_numeric(risk["AI_Risk_total"], errors="coerce")
risk["observed_eligible"] = parse_bool(risk["is_backtest_eligible_observed_sample"])
risk["strict_eligible"] = parse_bool(risk["is_backtest_eligible_strict"])
if risk["hour"].duplicated().any():
    raise ValueError("Risk index contains duplicate hourly timestamps")
if len(risk) > 1 and not risk["hour"].sort_values().diff().dropna().eq(pd.Timedelta(hours=1)).all():
    raise ValueError("Risk index is not a continuous hourly grid")

regular_hours = set(regular["bar_start_utc"].dt.tz_convert("America/New_York").dt.hour.unique())
bridge_hours = set(bridge["bar_start_utc"].dt.tz_convert("America/New_York").dt.hour.unique())
assert regular_hours == set(range(10, 16)), regular_hours
assert bridge_hours == {7, 8, 9}, bridge_hours
assert regular["duration_minutes"].eq(60).all()

targets = regular.loc[
    regular["window_complete"]
    & regular["ohlc_valid"]
    & ~regular["nonpositive_price"]
    & ~regular["split_event"]
    & ~regular["return_requires_review"]
    & regular["log_ret_oc"].notna()
].copy()
targets = targets.rename(columns={"log_ret_oc": "target_return"})
targets["target_start_ny"] = targets["bar_start_utc"].dt.tz_convert("America/New_York")
targets["target_end_ny"] = targets["bar_end_utc"].dt.tz_convert("America/New_York")
targets["target_hour"] = targets["target_start_ny"].dt.hour.astype(str).str.zfill(2) + ":00"
targets["trading_date"] = targets["trading_date"].astype(str)
targets["weekday"] = pd.to_datetime(targets["trading_date"]).dt.day_name()

return_grid = pd.concat(
    [
        bridge[["ticker", "trading_date", "bar_start_utc", "window_complete", "ohlc_valid", "nonpositive_price", "split_event", "return_requires_review", "log_ret_oc"]],
        regular[["ticker", "trading_date", "bar_start_utc", "window_complete", "ohlc_valid", "nonpositive_price", "split_event", "return_requires_review", "log_ret_oc"]],
    ],
    ignore_index=True,
)
usable_return = (
    return_grid["window_complete"]
    & return_grid["ohlc_valid"]
    & ~return_grid["nonpositive_price"]
    & ~return_grid["split_event"]
    & ~return_grid["return_requires_review"]
)
return_grid.loc[~usable_return, "log_ret_oc"] = np.nan
return_grid["trading_date"] = return_grid["trading_date"].astype(str)
if return_grid.duplicated(["ticker", "bar_start_utc"]).any():
    raise ValueError("Combined bridge/regular return grid has duplicate keys")

return_lookup = return_grid.set_index(["ticker", "bar_start_utc"])["log_ret_oc"]
date_lookup = return_grid.set_index(["ticker", "bar_start_utc"])["trading_date"]
risk_value_lookup = risk.set_index("hour")["AI_Risk_total"]
risk_eligible_lookup = risk.set_index("hour")["observed_eligible"]

for lag in range(1, 4):
    return_key_time = targets["bar_start_utc"] - pd.Timedelta(hours=lag)
    return_keys = pd.MultiIndex.from_arrays([targets["ticker"], return_key_time])
    targets[f"return_lag_{lag}"] = return_lookup.reindex(return_keys).to_numpy()
    lag_dates = date_lookup.reindex(return_keys).to_numpy()
    same_day = lag_dates == targets["trading_date"].to_numpy()
    targets.loc[~same_day, f"return_lag_{lag}"] = np.nan

    risk_hour = targets["bar_start_utc"] - pd.Timedelta(hours=lag)
    targets[f"risk_hour_lag_{lag}"] = risk_hour
    targets[f"risk_lag_{lag}"] = risk_value_lookup.reindex(risk_hour).to_numpy()
    eligible = risk_eligible_lookup.reindex(risk_hour)
    targets[f"risk_eligible_lag_{lag}"] = eligible.fillna(False).astype(bool).to_numpy()

targets = targets.sort_values(["bar_start_utc", "ticker"]).reset_index(drop=True)

assert (targets["risk_hour_lag_1"] + pd.Timedelta(hours=1) == targets["bar_start_utc"]).all()
assert not targets.duplicated(["ticker", "bar_start_utc"]).any()

audit = pd.DataFrame(
    [
        ("regular target rows", len(targets)),
        ("tickers", targets["ticker"].nunique()),
        ("regular trading dates", targets["trading_date"].nunique()),
        ("bridge expected rows", len(bridge)),
        ("bridge complete rows", int(bridge["window_complete"].sum())),
        ("risk hourly rows", len(risk)),
        ("risk observed-sample eligible rows", int(risk["observed_eligible"].sum())),
        ("risk strict-coverage eligible rows", int(risk["strict_eligible"].sum())),
        ("risk queries at result cap", int(pd.to_numeric(risk.get("queries_at_result_cap"), errors="coerce").max())),
    ],
    columns=["check", "value"],
)
display(audit)

example_row = targets.loc[
    targets["ticker"].eq("MSFT")
    & targets["target_start_ny"].dt.hour.eq(10)
    & targets["risk_lag_3"].notna()
].iloc[0]
alignment_example = pd.DataFrame(
    {
        "role": ["target return", "lag 1 risk/return", "lag 2 risk/return", "lag 3 risk/return"],
        "window_start_NY": [
            example_row["target_start_ny"],
            example_row["target_start_ny"] - pd.Timedelta(hours=1),
            example_row["target_start_ny"] - pd.Timedelta(hours=2),
            example_row["target_start_ny"] - pd.Timedelta(hours=3),
        ],
        "window_end_NY": [
            example_row["target_end_ny"],
            example_row["target_start_ny"],
            example_row["target_start_ny"] - pd.Timedelta(hours=1),
            example_row["target_start_ny"] - pd.Timedelta(hours=2),
        ],
    }
)
display(Markdown("**Exact no-look-ahead example (MSFT):**"))
display(alignment_example)

,check,value
0,regular target rows,2760
1,tickers,20
2,regular trading dates,23
3,bridge expected rows,1380
4,bridge complete rows,887
5,risk hourly rows,723
6,risk observed-sample eligible rows,622
7,risk strict-coverage eligible rows,0
8,risk queries at result cap,33


**Exact no-look-ahead example (MSFT):**

,role,window_start_NY,window_end_NY
0,target return,2026-09-02 10:00:00-04:00,2026-09-02 11:00:00-04:00
1,lag 1 risk/return,2026-09-02 09:00:00-04:00,2026-09-02 10:00:00-04:00
2,lag 2 risk/return,2026-09-02 08:00:00-04:00,2026-09-02 09:00:00-04:00
3,lag 3 risk/return,2026-09-02 07:00:00-04:00,2026-09-02 08:00:00-04:00


## 2. Construct the fixed-composition equal-weight portfolio

The regular target return requires all 20 stocks. The same rule is applied to bridge hours used as return lags. This avoids turning a changing subset of liquid premarket stocks into a different portfolio.

In [3]:
REQUESTED_TICKERS = (
    "MSFT", "GOOGL", "META", "AMZN", "TSLA", "ORCL", "CRM", "NOW", "ADBE", "WDAY",
    "INTU", "PATH", "AAPL", "PLTR", "UBER", "SNOW", "SHOP", "RBLX", "SNAP", "PINS",
)
EXPECTED_TICKERS = tuple(sorted(regular["ticker"].unique()))
if set(EXPECTED_TICKERS) != set(REQUESTED_TICKERS):
    missing = sorted(set(REQUESTED_TICKERS) - set(EXPECTED_TICKERS))
    extra = sorted(set(EXPECTED_TICKERS) - set(REQUESTED_TICKERS))
    raise ValueError(f"Ticker pool mismatch; missing={missing}, extra={extra}")


def build_equal_weight_hours(frame: pd.DataFrame, source_role: str) -> pd.DataFrame:
    work = frame.copy()
    work["ret_oc"] = pd.to_numeric(work["ret_oc"], errors="coerce")
    work["usable_component"] = (
        work["window_complete"]
        & work["ohlc_valid"]
        & ~work["nonpositive_price"]
        & ~work["split_event"]
        & ~work["return_requires_review"]
        & work["ret_oc"].notna()
        & work["duration_minutes"].eq(60)
    )
    rows = []
    keys = ["trading_date", "bar_start_utc", "bar_end_utc"]
    for (trading_date, bar_start, bar_end), group in work.groupby(keys, sort=True):
        usable = group.loc[group["usable_component"]].copy()
        usable_tickers = set(usable["ticker"])
        complete = usable_tickers == set(EXPECTED_TICKERS) and len(usable) == len(EXPECTED_TICKERS)
        simple_return = float(usable["ret_oc"].mean()) if complete else np.nan
        if complete and simple_return <= -1:
            raise ValueError(f"Invalid equal-weight return at {bar_start}")
        rows.append(
            {
                "source_role": source_role,
                "trading_date": str(trading_date),
                "bar_start_utc": bar_start,
                "bar_end_utc": bar_end,
                "n_valid_constituents": int(usable["ticker"].nunique()),
                "all_20_constituents_valid": bool(complete),
                "portfolio_ret_oc": simple_return,
                "portfolio_log_ret_oc": float(np.log1p(simple_return)) if complete else np.nan,
            }
        )
    result = pd.DataFrame(rows).sort_values("bar_start_utc").reset_index(drop=True)
    if result["bar_start_utc"].duplicated().any():
        raise ValueError(f"Duplicate portfolio timestamps for {source_role}")
    return result


regular_portfolio = build_equal_weight_hours(regular, "regular_target")
bridge_portfolio = build_equal_weight_hours(bridge, "bridge_lag_only")
if not regular_portfolio["all_20_constituents_valid"].all():
    raise ValueError("At least one regular portfolio target lacks a valid constituent")

portfolio_grid = pd.concat([bridge_portfolio, regular_portfolio], ignore_index=True)
portfolio_grid = portfolio_grid.sort_values("bar_start_utc").reset_index(drop=True)
if portfolio_grid["bar_start_utc"].duplicated().any():
    raise ValueError("Combined portfolio grid contains duplicate timestamps")

portfolio_targets = regular_portfolio.rename(columns={"portfolio_log_ret_oc": "target_return"}).copy()
portfolio_targets["target_start_ny"] = portfolio_targets["bar_start_utc"].dt.tz_convert("America/New_York")
portfolio_targets["target_end_ny"] = portfolio_targets["bar_end_utc"].dt.tz_convert("America/New_York")
portfolio_targets["target_hour"] = portfolio_targets["target_start_ny"].dt.hour.astype(str).str.zfill(2) + ":00"
portfolio_targets["weekday"] = pd.to_datetime(portfolio_targets["trading_date"]).dt.day_name()

portfolio_return_lookup = portfolio_grid.set_index("bar_start_utc")["portfolio_log_ret_oc"]
portfolio_date_lookup = portfolio_grid.set_index("bar_start_utc")["trading_date"]
for lag in range(1, 4):
    lag_start = portfolio_targets["bar_start_utc"] - pd.Timedelta(hours=lag)
    portfolio_targets[f"return_lag_{lag}"] = portfolio_return_lookup.reindex(lag_start).to_numpy()
    lag_dates = portfolio_date_lookup.reindex(lag_start).to_numpy()
    same_day = lag_dates == portfolio_targets["trading_date"].to_numpy()
    portfolio_targets.loc[~same_day, f"return_lag_{lag}"] = np.nan

    risk_hour = portfolio_targets["bar_start_utc"] - pd.Timedelta(hours=lag)
    portfolio_targets[f"risk_hour_lag_{lag}"] = risk_hour
    portfolio_targets[f"risk_lag_{lag}"] = risk_value_lookup.reindex(risk_hour).to_numpy()
    eligible = risk_eligible_lookup.reindex(risk_hour)
    portfolio_targets[f"risk_eligible_lag_{lag}"] = eligible.fillna(False).astype(bool).to_numpy()

assert (portfolio_targets["risk_hour_lag_1"] + pd.Timedelta(hours=1) == portfolio_targets["bar_start_utc"]).all()

portfolio_audit = pd.DataFrame(
    {
        "check": [
            "constituents", "regular target hours", "regular hours with all 20",
            "bridge hours", "bridge hours with all 20", "regular target dates",
        ],
        "value": [
            len(EXPECTED_TICKERS),
            len(regular_portfolio),
            int(regular_portfolio["all_20_constituents_valid"].sum()),
            len(bridge_portfolio),
            int(bridge_portfolio["all_20_constituents_valid"].sum()),
            regular_portfolio["trading_date"].nunique(),
        ],
    }
)
display(portfolio_audit)

bridge_constituent_counts = (
    bridge_portfolio.groupby(
        bridge_portfolio["bar_start_utc"].dt.tz_convert("America/New_York").dt.hour
    )["n_valid_constituents"]
    .agg(["min", "median", "max"])
    .rename_axis("bridge_start_hour_NY")
)
display(Markdown("### Valid constituent counts in bridge hours"))
display(bridge_constituent_counts)

,check,value
0,constituents,20
1,regular target hours,138
2,regular hours with all 20,138
3,bridge hours,69
4,bridge hours with all 20,0
5,regular target dates,23


### Valid constituent counts in bridge hours

,min,median,max
bridge_start_hour_NY,,,
7,10,14,17
8,11,14,19
9,9,10,11


## 3. Select portfolio return lag order $q$ by BIC

All three BIC candidates use the same rows with valid portfolio return lags through three hours. Lower BIC is preferred; if another order is within 2 BIC points, the notebook flags the near tie.

In [4]:
def portfolio_fixed_effects(frame: pd.DataFrame) -> pd.DataFrame:
    categories = frame[["target_hour", "weekday"]].astype(str)
    return pd.get_dummies(
        categories,
        columns=["target_hour", "weekday"],
        prefix=["hour", "weekday"],
        drop_first=True,
        dtype=float,
    )


def portfolio_design(frame: pd.DataFrame, q: int, p: int = 0) -> pd.DataFrame:
    matrix = pd.DataFrame({"const": np.ones(len(frame), dtype=float)}, index=frame.index)
    for lag in range(1, q + 1):
        matrix[f"return_lag_{lag}"] = frame[f"return_lag_{lag}"].astype(float)
    for lag in range(1, p + 1):
        matrix[f"risk_lag_{lag}"] = frame[f"risk_lag_{lag}"].astype(float)
    return pd.concat([matrix, portfolio_fixed_effects(frame)], axis=1).astype(float)


bic_mask = portfolio_targets["target_return"].notna()
for lag in range(1, 4):
    bic_mask &= portfolio_targets[f"return_lag_{lag}"].notna()
portfolio_bic_sample = portfolio_targets.loc[bic_mask].sort_values("bar_start_utc").copy()

bic_rows = []
for q in (1, 2, 3):
    X = portfolio_design(portfolio_bic_sample, q=q, p=0)
    if np.linalg.matrix_rank(X.to_numpy()) != X.shape[1]:
        raise ValueError(f"Rank-deficient portfolio BIC design for q={q}")
    fit = sm.OLS(portfolio_bic_sample["target_return"].astype(float), X, missing="raise").fit()
    bic_rows.append(
        {
            "q_return_lags": q,
            "n_obs_common_sample": int(fit.nobs),
            "n_parameters": int(fit.df_model + 1),
            "BIC": float(fit.bic),
            "adjusted_R2": float(fit.rsquared_adj),
        }
    )

portfolio_bic_results = pd.DataFrame(bic_rows).sort_values("q_return_lags").reset_index(drop=True)
portfolio_bic_results["delta_BIC"] = portfolio_bic_results["BIC"] - portfolio_bic_results["BIC"].min()
Q_SELECTED = int(
    portfolio_bic_results.sort_values(["BIC", "q_return_lags"]).iloc[0]["q_return_lags"]
)
portfolio_bic_results["selected"] = portfolio_bic_results["q_return_lags"].eq(Q_SELECTED)

display(
    portfolio_bic_results.style.format(
        {"BIC": "{:.3f}", "delta_BIC": "{:.3f}", "adjusted_R2": "{:.6f}"}
    ).hide(axis="index")
)
display(Markdown(f"**Selected portfolio return lag order: q={Q_SELECTED}.**"))
near_ties = portfolio_bic_results.loc[
    (portfolio_bic_results["delta_BIC"] > 0) & (portfolio_bic_results["delta_BIC"] < 2)
]
if not near_ties.empty:
    labels = ", ".join(
        f"q={int(row.q_return_lags)} (ΔBIC={row.delta_BIC:.3f})"
        for row in near_ties.itertuples()
    )
    display(Markdown(f"**Near-tied alternatives: {labels}.**"))

q_return_lags,n_obs_common_sample,n_parameters,BIC,adjusted_R2,delta_BIC,selected
1,69,8,-601.231,-0.054254,2.411,False
2,69,9,-597.130,-0.069763,6.512,False
3,69,10,-603.642,0.068997,0.000,True


**Selected portfolio return lag order: q=3.**

## 4. Equal-weight portfolio Granger tests

The restricted and unrestricted models use the same observations, and $p=1,2,3$ all use one common sample with valid risk lags through three hours.

$$r^{EW}_{t}=\alpha+\sum_{j=1}^{q}\phi_jr^{EW}_{t-j}+\sum_{j=1}^{p}\beta_jRisk_{t-j}+\text{hour FE}+\text{weekday FE}+\epsilon_t$$

The null is $H_0:\beta_1=\cdots=\beta_p=0$.

In [5]:
RISK_SD = float(risk.loc[risk["observed_eligible"], "AI_Risk_total"].std(ddof=1))
if not np.isfinite(RISK_SD) or RISK_SD <= 0:
    raise ValueError("AI_Risk_total has no usable variation")

main_mask = portfolio_targets["target_return"].notna()
for lag in range(1, Q_SELECTED + 1):
    main_mask &= portfolio_targets[f"return_lag_{lag}"].notna()
for lag in range(1, 4):
    main_mask &= (
        portfolio_targets[f"risk_lag_{lag}"].notna()
        & portfolio_targets[f"risk_eligible_lag_{lag}"]
    )
PORTFOLIO_COMMON_SAMPLE = portfolio_targets.loc[main_mask].sort_values("bar_start_utc").reset_index(drop=True)
if PORTFOLIO_COMMON_SAMPLE.empty:
    raise ValueError("No common portfolio regression sample")


def fit_portfolio_model(sample: pd.DataFrame, q: int, p: int) -> dict:
    y = sample["target_return"].astype(float)
    X_restricted = portfolio_design(sample, q=q, p=0)
    X_unrestricted = portfolio_design(sample, q=q, p=p)
    if len(sample) <= X_unrestricted.shape[1] + 20:
        raise ValueError(f"Insufficient portfolio residual degrees of freedom for p={p}")
    if np.linalg.matrix_rank(X_unrestricted.to_numpy()) != X_unrestricted.shape[1]:
        raise ValueError(f"Rank-deficient portfolio design for p={p}")

    restricted = sm.OLS(y, X_restricted, missing="raise").fit()
    unrestricted = sm.OLS(y, X_unrestricted, missing="raise").fit()
    names = list(X_unrestricted.columns)
    risk_positions = [names.index(f"risk_lag_{lag}") for lag in range(1, p + 1)]
    restriction = np.zeros((p, len(names)), dtype=float)
    for row_number, position in enumerate(risk_positions):
        restriction[row_number, position] = 1.0

    hac = unrestricted.get_robustcov_results(
        cov_type="HAC",
        maxlags=5,
        use_correction=True,
    )
    hac_wald = hac.wald_test(restriction, use_f=True, scalar=True)

    day_groups = pd.Categorical(sample["trading_date"]).codes
    day_clustered = unrestricted.get_robustcov_results(
        cov_type="cluster",
        groups=day_groups,
        use_correction=True,
        df_correction=True,
    )
    day_wald = day_clustered.wald_test(restriction, use_f=True, scalar=True)
    classical_f, classical_f_p, _ = unrestricted.compare_f_test(restricted)

    params = np.asarray(day_clustered.params, dtype=float)
    covariance = np.asarray(day_clustered.cov_params(), dtype=float)
    sum_vector = np.zeros(len(names), dtype=float)
    sum_vector[risk_positions] = 1.0
    beta_sum = float(sum_vector @ params)
    beta_sum_variance = float(sum_vector @ covariance @ sum_vector)
    beta_sum_se = np.sqrt(beta_sum_variance) if beta_sum_variance >= 0 else np.nan
    inference_df = float(day_wald.df_denom)
    critical_value = float(stats.t.ppf(0.975, inference_df))
    ci_low = beta_sum - critical_value * beta_sum_se if np.isfinite(beta_sum_se) else np.nan
    ci_high = beta_sum + critical_value * beta_sum_se if np.isfinite(beta_sum_se) else np.nan

    return {
        "risk_lags_p": p,
        "return_lags_q": q,
        "n_obs": int(unrestricted.nobs),
        "n_trading_days": int(sample["trading_date"].nunique()),
        "n_parameters": int(unrestricted.df_model + 1),
        "residual_df": int(unrestricted.df_resid),
        "risk_beta_sum": beta_sum,
        "effect_1SD_percent": 100 * np.expm1(beta_sum * RISK_SD),
        "effect_1SD_CI_low_percent": 100 * np.expm1(ci_low * RISK_SD),
        "effect_1SD_CI_high_percent": 100 * np.expm1(ci_high * RISK_SD),
        "HAC5_F": float(np.asarray(hac_wald.statistic).squeeze()),
        "HAC5_p": float(np.asarray(hac_wald.pvalue).squeeze()),
        "day_cluster_F": float(np.asarray(day_wald.statistic).squeeze()),
        "day_cluster_df_denom": float(day_wald.df_denom),
        "day_cluster_p": float(np.asarray(day_wald.pvalue).squeeze()),
        "classical_F": float(classical_f),
        "classical_F_p": float(classical_f_p),
        "adjusted_R2": float(unrestricted.rsquared_adj),
    }


portfolio_results = pd.DataFrame(
    [fit_portfolio_model(PORTFOLIO_COMMON_SAMPLE, q=Q_SELECTED, p=p) for p in (1, 2, 3)]
)
portfolio_results["BH_q_value"] = multipletests(portfolio_results["day_cluster_p"], method="fdr_bh")[1]
portfolio_results["FDR_5pct_significant"] = portfolio_results["BH_q_value"] < 0.05

portfolio_display = portfolio_results[[
    "risk_lags_p", "return_lags_q", "n_obs", "n_trading_days", "risk_beta_sum",
    "effect_1SD_percent", "effect_1SD_CI_low_percent", "effect_1SD_CI_high_percent",
    "day_cluster_F", "day_cluster_df_denom", "day_cluster_p", "BH_q_value",
    "FDR_5pct_significant", "HAC5_p", "adjusted_R2",
]].rename(
    columns={
        "risk_lags_p": "风险滞后p",
        "return_lags_q": "收益滞后q",
        "n_obs": "观测数",
        "n_trading_days": "交易日数",
        "risk_beta_sum": "风险系数合计",
        "effect_1SD_percent": "风险+1SD组合收益影响(%)",
        "effect_1SD_CI_low_percent": "95%CI下限(%)",
        "effect_1SD_CI_high_percent": "95%CI上限(%)",
        "day_cluster_F": "按交易日聚类联合F",
        "day_cluster_df_denom": "分母df",
        "day_cluster_p": "原始p值",
        "BH_q_value": "BH校正q值",
        "FDR_5pct_significant": "FDR 5%显著",
        "HAC5_p": "HAC(5个观测滞后)p值",
        "adjusted_R2": "调整R²",
    }
)

display(Markdown("### 等权组合主回归结果"))
display(
    portfolio_display.style.format(
        {
            "风险系数合计": "{:.8f}",
            "风险+1SD组合收益影响(%)": "{:.5f}",
            "95%CI下限(%)": "{:.5f}",
            "95%CI上限(%)": "{:.5f}",
            "按交易日聚类联合F": "{:.4f}",
            "分母df": "{:.0f}",
            "原始p值": "{:.6g}",
            "BH校正q值": "{:.6g}",
            "HAC(5个观测滞后)p值": "{:.6g}",
            "调整R²": "{:.6f}",
        }
    ).hide(axis="index")
)

sample_by_hour = PORTFOLIO_COMMON_SAMPLE.groupby("target_hour").size().rename("observations").to_frame()
display(Markdown("### 共同样本的目标时段构成"))
display(sample_by_hour)

significant = portfolio_results.loc[portfolio_results["FDR_5pct_significant"]]
if significant.empty:
    conclusion = (
        "Trading-date clustered inference and BH correction do not reject the no-Granger-predictability null "
        "for any of p=1,2,3."
    )
else:
    labels = ", ".join(f"p={int(value)}" for value in significant["risk_lags_p"])
    conclusion = f"BH-adjusted 5% significance is present for: {labels}."
display(Markdown("### Mechanical conclusion"))
display(Markdown(conclusion))

### 等权组合主回归结果

风险滞后p,收益滞后q,观测数,交易日数,风险系数合计,风险+1SD组合收益影响(%),95%CI下限(%),95%CI上限(%),按交易日聚类联合F,分母df,原始p值,BH校正q值,FDR 5%显著,HAC(5个观测滞后)p值,调整R²
1,3,54,18,0.00022920,0.03248,-0.05159,0.11662,0.6640,17,0.426428,0.858119,False,0.418609,0.081433
2,3,54,18,0.00012326,0.01746,-0.06729,0.10229,0.3689,17,0.696875,0.858119,False,0.714303,0.065921
3,3,54,18,0.00021674,0.03071,-0.09290,0.15447,0.2530,17,0.858119,0.858119,False,0.874739,0.046335


### 共同样本的目标时段构成

,observations
target_hour,
13:00,18
14:00,18
15:00,18


### Mechanical conclusion

Trading-date clustered inference and BH correction do not reject the no-Granger-predictability null for any of p=1,2,3.

## 5. Interpretation limits

1. The fixed 20-stock composition is maintained. Because at least one constituent lacks a complete return in every 07:00–10:00 bridge hour, those bridge portfolio returns remain missing; early target hours that require them are excluded rather than calculated from a changing subset.
2. The X News collection hit every query result cap and has zero strictly complete-coverage hours. Results describe only the observed X News sample.
3. The sample is short and contains hour/weekday controls. Non-significance does not prove the economic effect is exactly zero.
4. For p>1, the reported +1 SD cumulative effect means all included risk lags rise by one standard deviation together; it is not a one-hour impulse response.
5. The equal-weight result answers an aggregate basket question. It can differ from individual-stock results when stock responses have different signs or magnitudes.